## setting up log

In [10]:
import logging
import os
import time

import psycopg2
from psycopg2 import OperationalError, Error as Psycopg2Error
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv

logging.basicConfig(
    filename="w2d3.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s |%(message)s"
)
logger = logging.getLogger("intership_w2d3")
print("Logging configured -> w2d3.log")

Logging configured -> w2d3.log


In [4]:
logger.info("Week 2 day 3 - Implementing Normalization started")

In [8]:
import pandas as pf
filepath = "data/tiktok.csv"

df = pf.read_csv(filepath)


logger.info("Dataframe loaded from file: %s", filepath)
print(df.shape)
df.head()

(800, 9)


,Comment ID,Reply to Which Comment,User ID,Comment,Comment Time,Digg Count,Author Digged,Reply Count,Pinned to Top
0,7669319408842081044,NaN,6602052900607623174,🥰🥰🥰,"8/2/2026, 12:21:30 PM",1,Yes,0.0,No
1,7669319064204641044,NaN,6602052900607623174,🥰🥰🥰,"8/2/2026, 12:20:13 PM",1,Yes,0.0,No
2,7669318657889272596,NaN,6602052900607623174,🥰🥰🥰,"8/2/2026, 12:18:36 PM",1,Yes,0.0,No
3,7666354577964712720,NaN,7410829670142657542,❤️❤️❤️,"7/25/2026, 12:36:20 PM",1,Yes,0.0,No
4,7665424955869905685,NaN,6555001822149350400,🙌🙌,"7/23/2026, 12:28:58 AM",1,Yes,1.0,No


In [13]:
print(df.columns.tolist())

['Comment ID', 'Reply to Which Comment', 'User ID', 'Comment', 'Comment Time', 'Digg Count', 'Author Digged', 'Reply Count', 'Pinned to Top']


In [14]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Comment ID              800 non-null    int64  
 1   Reply to Which Comment  369 non-null    float64
 2   User ID                 800 non-null    int64  
 3   Comment                 788 non-null    str    
 4   Comment Time            800 non-null    str    
 5   Digg Count              800 non-null    int64  
 6   Author Digged           800 non-null    str    
 7   Reply Count             431 non-null    float64
 8   Pinned to Top           431 non-null    str    
dtypes: float64(2), int64(3), str(4)
memory usage: 56.4 KB
None


## Loading Credentials


In [11]:
load_dotenv(override=True)  # looks for a ".env" file relative to this notebook

DB_HOST = os.getenv("DB_HOST")       # e.g. "127.0.0.1" (placeholder shown here)
DB_PORT = os.getenv("DB_PORT")       # e.g. "5432"
DB_NAME = os.getenv("DB_NAME")       # e.g. "banking_etl"
DB_USER = os.getenv("DB_USER")       # e.g. "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")  # never printed or committed

_required = {"DB_HOST": DB_HOST, "DB_PORT": DB_PORT, "DB_NAME": DB_NAME,
             "DB_USER": DB_USER, "DB_PASSWORD": DB_PASSWORD}
_missing = [k for k, v in _required.items() if not v]
if _missing:
    raise RuntimeError(
        f"Missing required .env variable(s): {', '.join(_missing)}. "
        f"Please check your .env file."
    )
logger.info("Credentials loaded from .env")
print("Credentials loaded from .env")

Credentials loaded from .env


In [12]:
class DBConnection:
    """Small wrapper so every part of the pipeline shares one connection."""

    def __init__(self, host, port, dbname, user, password):
        self.host, self.port, self.dbname, self.user, self.password = (
            host, port, dbname, user, password
        )
        self._connect()

    def _connect(self):
        self.conn = psycopg2.connect(
            host=self.host, port=self.port, dbname=self.dbname,
            user=self.user, password=self.password,
        )
        logger.info("Connected to database '%s'", self.dbname)

    def reconnect(self):
        """Used by the retry decorator when the connection itself has died."""
        try:
            self.conn.close()
        except Exception:
            pass
        self._connect()
        logger.info("Reconnected to database '%s'", self.dbname)

    def cursor(self):
        return self.conn.cursor(cursor_factory=RealDictCursor)

    def commit(self):
        self.conn.commit()

    def rollback(self):
        self.conn.rollback()

    def close(self):
        self.conn.close()
        logger.info("Connection closed")


def with_retry(max_attempts=3, delay_seconds=1):
    """Retries a db-operation function up to max_attempts times on
    psycopg2.OperationalError, always rolling back (or reconnecting)
    before each retry so the connection never sits in a broken state.
    The decorated function's first positional argument must be the db object.
    """
    def decorator(func):
        def wrapper(db, *args, **kwargs):
            attempt = 0
            while True:
                attempt += 1
                try:
                    return func(db, *args, **kwargs)
                except OperationalError as e:
                    logger.error(
                        "Attempt %d/%d for %s failed: %s",
                        attempt, max_attempts, func.__name__, e,
                    )
                    try:
                        db.rollback()
                    except Exception:
                        # Connection itself is dead (not just the transaction) - reconnect.
                        try:
                            db.reconnect()
                        except Exception as reconnect_err:
                            logger.error("Reconnect failed: %s", reconnect_err)
                    if attempt >= max_attempts:
                        logger.error(
                            "Giving up on %s after %d attempts", func.__name__, max_attempts
                        )
                        raise
                    time.sleep(delay_seconds)
        return wrapper
    return decorator


db = DBConnection(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
print("Connected")


Connected


In [16]:
DDL_TABLES = """
CREATE TABLE tiktok_comments (
    comment_id BIGINT PRIMARY KEY,
    reply_to_which_comment BIGINT,
    user_id BIGINT NOT NULL,
    comment TEXT,
    comment_time TIMESTAMP,
    digg_count INTEGER,
    author_digged VARCHAR(10),
    reply_count INTEGER,
    pinned_to_top VARCHAR(10)
);
"""


@with_retry()
def create_schema(db):
    try:
        with db.cursor() as cur:
            cur.execute(DDL_TABLES)
        db.commit()
        logger.info("Schema created/verified (tiktok_comments table)")
        print("Schema created/verified")
    except Psycopg2Error as e:
        db.rollback()
        logger.error("Schema creation failed: %s", e)
        raise


create_schema(db)

Schema created/verified


In [17]:
def read_data(db):
    try:
        with db.cursor() as cur:
            cur.execute("SELECT * FROM tiktok_comments")

            rows = cur.fetchall()

            for row in rows:
                print(row)

    except Psycopg2Error as e:
        logger.error("Failed to read data: %s", e)
        raise


read_data(db)